In [ ]:
from flappy_agent import *
import numpy as np

In [ ]:
# Fresh agent
agent = FlappyAgent([1024, 128, 16,256,256])
# Or load existing weights to continue training:
#w_path='flappy weights/flappy_ppo_64x64x64.pt'
#agent = FlappyAgent(hidden_layers=[1024, 128, 16,256,256],weights_path=w_path)

In [ ]:
training_params = {
    'gamma': 0.99,
    'lam': 0.95,
    'clip_eps': 0.2,
    'clip_coef': 1.0,
    'value_coef': 0.5,
    'entropy_coef': 0.01,
    'max_grad_norm': 5.0,
    'learning_rate': 0.0003,
    'ppo_epochs': 5,
    'num_epochs': 5000,
    'target_steps': 8192,
    'minibatch_size': 512,
    'print_freq': 10,
    'ema_alpha': 0.9,
    'value_loss': 'mse',
    'normalize_advantage': True,
    'reward_values': {'positive':1.0, 'tick':0.0, 'loss':-5.0},
}

In [ ]:
print('Using parameters:')
for key, value in training_params.items():
    print(f'{key}: {value}')
print()
agent.run_training(**training_params, test_exploit=True, result_path='Results/NN32x32x32.csv')

In [ ]:
#torch.save(agent.network.state_dict(), 'flappy weights/flappy_ppo_64x64x64.pt')
print("Weights saved to 'flappy weights/flappy_ppo_trained2.pt'")

In [ ]:
from ple.games.flappybird import FlappyBird
from ple import PLE
game = FlappyBird()
episode = PLE(game, fps=30, display_screen=False, force_fps=True)
episode.init()

agent.play_episode(episode, mode='Exploit', max_pipes=None, print_freq=10000)

In [ ]:
#print(states)
for col_name, col_data in states.items():
    mean_val = np.mean(np.array(col_data))
    std_val = np.std(np.array(col_data))
    print(f'{col_name}: {mean_val:.3f} +/- {std_val:.3f}')

In [ ]:
import numpy as np

num_test_episodes = 10
scores = []

from ple.games.flappybird import FlappyBird
from ple import PLE
game = FlappyBird()
episode = PLE(game, fps=30, display_screen=False, force_fps=True)
episode.init()

for ep in range(num_test_episodes):
    agent.play_episode(episode, mode='Exploit', print_freq=1000)
    score = agent.memory[:, 9].sum().item()  # reward is at index 9 (8 state features + action + reward)
    scores.append(score)
    print(f'\nEpisode {ep+1}, score {score}')

scores = np.array(scores)
print(f"Average score: {scores.mean():.1f} +/- {scores.std():.1f}")
print(f"Max score: {scores.max():.1f}")
print(f"Min score: {scores.min():.1f}")

In [ ]:
import pygame
import pygame
import os

os.environ.pop('SDL_VIDEODRIVER', None)
pygame.quit()
pygame.init()

num_runs = 10
game_vis = FlappyBird()
p_vis = PLE(game_vis, fps=30, display_screen=True, force_fps=False)
p_vis.init()

for run in range(num_runs):
    p_vis.reset_game()
    total_reward = 0
    while not p_vis.game_over():
        state = state_to_tensor(p_vis.getGameState())
        action, _, _ = agent.get_action(state, mode='Exploit')
        reward = p_vis.act(p_vis.getActionSet()[action])
        total_reward += reward
    print(f"Run {run+1}/{num_runs} - Total reward: {total_reward:.1f}")

pygame.quit()
print("Done.")